In [1]:
from functools import lru_cache
from itertools import combinations_with_replacement


def canonical(heaps):
    return tuple(sorted(heaps))


def nim_sum(heaps):
    result = 0
    for heap in heaps:
        result ^= heap
    return result

@lru_cache(maxsize=None)
def nim_win(heaps):
    heaps = canonical(heaps)

    if sum(heaps) == 0:
        return True
    for i in range(len(heaps)):
        heap_size = heaps[i]
        for number_removed in range(1, heap_size + 1):
            new_heaps = list(heaps)
            new_heaps[i] -= number_removed
            new_heaps = canonical(new_heaps)
            if not nim_win(new_heaps):
                return True
    return False

def nim_sum_prediction(heaps):
    positive_heaps = [
        heap for heap in heaps
        if heap > 0
    ]

    if len(positive_heaps) == 0:
        return True

    if all(heap == 1 for heap in positive_heaps):
        return len(positive_heaps) % 2 == 0

    return nim_sum(positive_heaps) != 0


def check_nim_sum_conjecture(max_heaps=5, max_stones=12):
    mismatches = []

    for n in range(1, max_heaps + 1):

        possible_sizes = range(max_stones + 1)

        for heaps in combinations_with_replacement(
            possible_sizes, n
        ):

            if sum(heaps) > max_stones:
                continue

            recursive_answer = nim_win(heaps)

            predicted_answer = nim_sum_prediction(heaps)

            if recursive_answer != predicted_answer:

                mismatches.append(
                    (
                        heaps,
                        recursive_answer,
                        predicted_answer
                    )
                )

    return mismatches

@lru_cache(maxsize=None)
def build_win(heaps, remaining):
    heaps = canonical(heaps)
    if remaining == 0:
        return nim_win(heaps)
    next_states = set()
    for i in range(len(heaps)):
        new_heaps = list(heaps)
        new_heaps[i] += 1
        new_heaps = canonical(new_heaps)
        next_states.add(new_heaps)
        
    for new_heaps in next_states:

        if not build_win(
            new_heaps,
            remaining - 1
        ):
            return True

    return False


def winner(n, s):
    initial_heaps = tuple([0] * n)
    if build_win(initial_heaps, s):
        return "P1"
    return "P2"

def winning_build_moves(heaps, remaining):
    heaps = canonical(heaps)
    if remaining == 0:
        return []
    possible_states = set()
    for i in range(len(heaps)):
        new_heaps = list(heaps)
        new_heaps[i] += 1
        new_heaps = canonical(new_heaps)
        possible_states.add(new_heaps)
    winning_moves = []

    for new_heaps in possible_states:
        if not build_win(
            new_heaps,
            remaining - 1
        ):

            winning_moves.append(new_heaps)

    return sorted(winning_moves)


print("Small misere-Nim positions")
print("--------------------------")

test_positions = [
    (1,),
    (1, 1),
    (1, 1, 1),
    (2, 2),
    (1, 1, 2)
]

for position in test_positions:

    print(
        position,
        "nim sum =",
        nim_sum(position),
        "winning =",
        nim_win(position)
    )

print("\nChecking Nim-sum conjecture")
print("---------------------------")

mismatches = check_nim_sum_conjecture(
    max_heaps=5,
    max_stones=12
)

print("Number of mismatches:", len(mismatches))

if len(mismatches) > 0:

    print("Mismatches:")

    for mismatch in mismatches:
        print(mismatch)

print("\nWinner table")
print("------------")

print("      ", end="")

for s in range(1, 13):
    print(f"{s:>4}", end="")

print()

for n in range(2, 6):

    print(f"n={n}: ", end="")

    for s in range(1, 13):

        print(
            f"{winner(n, s):>4}",
            end=""
        )

    print()



print("\nExample (n, s) = (2, 4)")
print("-----------------------")

print("Winner:", winner(2, 4))


print("\nExample (n, s) = (3, 4)")
print("-----------------------")

print("Winner:", winner(3, 4))


print("\nWinning building moves")
print("----------------------")

examples = [
    ((0, 1), 3),
    ((1, 2), 1),
    ((0, 0, 2), 2),
    ((0, 1, 1), 2)
]

for heaps, remaining in examples:

    print(
        "State:",
        heaps,
        "| remaining:",
        remaining,
        "| winning moves:",
        winning_build_moves(
            heaps,
            remaining
        )
    )

Small misere-Nim positions
--------------------------
(1,) nim sum = 1 winning = False
(1, 1) nim sum = 0 winning = True
(1, 1, 1) nim sum = 1 winning = False
(2, 2) nim sum = 0 winning = False
(1, 1, 2) nim sum = 2 winning = True

Checking Nim-sum conjecture
---------------------------
Number of mismatches: 0

Winner table
------------
         1   2   3   4   5   6   7   8   9  10  11  12
n=2:   P1  P1  P2  P2  P2  P2  P2  P2  P2  P2  P2  P2
n=3:   P1  P1  P2  P1  P2  P2  P2  P1  P2  P1  P2  P1
n=4:   P1  P1  P2  P1  P2  P2  P2  P2  P2  P2  P2  P2
n=5:   P1  P1  P2  P1  P2  P2  P2  P2  P2  P1  P2  P1

Example (n, s) = (2, 4)
-----------------------
Winner: P2

Example (n, s) = (3, 4)
-----------------------
Winner: P1

Winning building moves
----------------------
State: (0, 1) | remaining: 3 | winning moves: [(1, 1)]
State: (1, 2) | remaining: 1 | winning moves: [(2, 2)]
State: (0, 0, 2) | remaining: 2 | winning moves: [(0, 0, 3)]
State: (0, 1, 1) | remaining: 2 | winning moves: [(1